In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/odinaka60/PPE-detection.git
%cd PPE-detection

In [ ]:
!pip install ultralytics roboflow

In [ ]:
from google.colab import userdata
import yaml

from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("roboflow-universe-projects").project("personal-protective-equipment-combined-model")
version = project.version(8)
dataset = version.download("yolov8")

with open(f"{dataset.location}/data.yaml") as f:
    print("Classes:", yaml.safe_load(f)["names"])            

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolov8s.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    device="0",
    patience=15,
    project="/content/runs/train",
    name="ppe_v1",
    save_period=10,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
print("Training complete. Best weights:", best_pt)

In [ ]:
import shutil
from pathlib import Path

save_dir = Path("/content/drive/MyDrive/PPE-detection/models")
save_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_pt, save_dir / "best.pt")

In [ ]:
best_model = YOLO(best_pt)
best_model.export(format="onnx", imgsz=640)
best_onnx = best_pt.with_suffix(".onnx")         
shutil.copy(best_onnx, save_dir / "best.onnx")

print("Saved to Drive:", save_dir / "best.pt", "and", save_dir / "best.onnx")